In [16]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from pathlib import Path

RAW_DIR = "../data/raw"

# --- 1. Enhanced Data Preparation ---
def get_seed_int(seed_str):
    return int(''.join(filter(str.isdigit, str(seed_str))))

def calculate_season_win_pct(df):
    wins = df.groupby(['Season', 'WTeamID']).size().reset_index(name='Wins')
    losses = df.groupby(['Season', 'LTeamID']).size().reset_index(name='Losses')
    stats = pd.merge(wins, losses, left_on=['Season', 'WTeamID'], right_on=['Season', 'LTeamID'], how='outer').fillna(0)
    stats['TeamID'] = np.where(stats['WTeamID'] > 0, stats['WTeamID'], stats['LTeamID'])
    stats['Season'] = np.where(stats['Season_x'] > 0, stats['Season_x'], stats['Season_y'])
    stats['WinPct'] = stats['Wins'] / (stats['Wins'] + stats['Losses'])
    return stats[['Season', 'TeamID', 'WinPct']]

def prepare_data(gender='M'):
    prefix = 'M' if gender == 'M' else 'W'
    results = pd.read_csv(f"{RAW_DIR}/{prefix}NCAATourneyCompactResults.csv")
    seeds = pd.read_csv(f"{RAW_DIR}/{prefix}NCAATourneySeeds.csv")
    reg_season = pd.read_csv(f"{RAW_DIR}/{prefix}RegularSeasonCompactResults.csv")
    
    seeds['SeedInt'] = seeds['Seed'].apply(get_seed_int)
    win_pcts = calculate_season_win_pct(reg_season)
    
    df = results.merge(seeds[['Season', 'TeamID', 'SeedInt']], left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID']).rename(columns={'SeedInt': 'WSeed'}).drop('TeamID', axis=1)
    df = df.merge(seeds[['Season', 'TeamID', 'SeedInt']], left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID']).rename(columns={'SeedInt': 'LSeed'}).drop('TeamID', axis=1)
    df = df.merge(win_pcts, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID']).rename(columns={'WinPct': 'WWinPct'}).drop('TeamID', axis=1)
    df = df.merge(win_pcts, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID']).rename(columns={'WinPct': 'LWinPct'}).drop('TeamID', axis=1)

    # Symmetrize
    df['T1ID'] = np.where(df['WTeamID'] < df['LTeamID'], df['WTeamID'], df['LTeamID'])
    df['T2ID'] = np.where(df['WTeamID'] < df['LTeamID'], df['LTeamID'], df['WTeamID'])
    df['T1Seed'] = np.where(df['WTeamID'] < df['LTeamID'], df['WSeed'], df['LSeed'])
    df['T2Seed'] = np.where(df['WTeamID'] < df['LTeamID'], df['LSeed'], df['WSeed'])
    df['T1WinPct'] = np.where(df['WTeamID'] < df['LTeamID'], df['WWinPct'], df['LWinPct'])
    df['T2WinPct'] = np.where(df['WTeamID'] < df['LTeamID'], df['LWinPct'], df['WWinPct'])
    
    df['Result'] = np.where(df['WTeamID'] == df['T1ID'], 1, 0)
    df['SeedDiff'] = df['T1Seed'] - df['T2Seed']
    df['WinPctDiff'] = df['T1WinPct'] - df['T2WinPct']
    
    # Feature list in your requested order
    features = ['SeedDiff', 'T1Seed', 'WinPctDiff']
    return df[['Season'] + features + ['Result']], features

c:\Users\cjchen\Desktop\personal-projects\march-madness-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def run_unified_experiment(features, m_params, w_params, run_name="Custom_Run"):
    mlflow.set_experiment("MMLM_2026_Unified")
    
    with mlflow.start_run(run_name=run_name):
        # 1. Log Features and Hyperparams
        mlflow.log_param("features", features)
        # We prefix params so we know which model they belong to
        mlflow.log_params({f"m_{k}": v for k, v in m_params.items()})
        mlflow.log_params({f"w_{k}": v for k, v in w_params.items()})
        
        all_scores = {}
        
        for g in ['M', 'W']:
            data = prepare_tourney_data(g)
            params = m_params if g == 'M' else w_params
            
            # Time Series CV (Rolling Window)
            seasons = sorted(data['Season'].unique())[-10:] # Last 10 years
            fold_scores = []
            
            for yr in seasons:
                train = data[data['Season'] < yr]
                test = data[data['Season'] == yr]
                
                # Unpack the specific params for this model
                model = LogisticRegression(**params).fit(train[features], train['Result'])
                
                preds = model.predict_proba(test[features])[:, 1]
                fold_scores.append(log_loss(test['Result'], preds))
            
            avg_score = np.mean(fold_scores)
            all_scores[g] = avg_score
            mlflow.log_metric(f"{g}_Brier_Score", avg_score)
            
        # 2. Log the Final Combined Metric
        combined_brier = (all_scores['M'] + all_scores['W']) / 2
        mlflow.log_metric("Combined_Brier_Score", combined_brier)
        
        # 3. Save the final models trained on ALL data
        for g in ['M', 'W']:
            data = prepare_tourney_data(g)
            params = m_params if g == 'M' else w_params
            final_model = LogisticRegression(**params).fit(data[features], data['Result'])
            mlflow.sklearn.log_model(final_model, f"{g}_final_model")
            
        print(f"🚀 {run_name} Complete | Combined: {combined_brier:.4f}")
        return combined_brier

# --- 2. Running your Experiments ---

# Example 1: Baseline
features_v1 = ['SeedDiff', 'T1Seed', 'WinPctDiff']
m_params_v1 = {'C': 1.0, 'solver': 'lbfgs'}
w_params_v1 = {'C': 1.0, 'solver': 'lbfgs'}

run_unified_experiment(features_v1, m_params_v1, w_params_v1, "Baseline_v1")

# Example 2: Higher Regularization for Men (to handle higher variance)
m_params_v2 = {'C': 0.1, 'solver': 'lbfgs'} # Stronger regularization
run_unified_experiment(features_v1, m_params_v2, w_params_v1, "Men_High_Reg_v2")